# Financial Market Regime & Event Intelligence Engine
## Notebook 02: Hidden Markov Model (HMM) Market Regime Detection

Welcome to Notebook 02! Having established a baseline K-Means clustering model in Notebook 01, we now implement a **Gaussian Hidden Markov Model (HMM)** to model financial market regimes as hidden temporal states.

### Objectives:
1. **Reproduce Data & Features**: Fetch S&P 500 (`^GSPC`), Long-Term Treasuries (`TLT`), and Gold (`GLD`) data to construct the 6-feature quantitative dataset.
2. **Standardize Features**: Rescale feature distributions using `StandardScaler` while preserving the date index.
3. **Fit Gaussian HMM**: Train a 4-state Gaussian HMM (`n_components=4`, `covariance_type="full"`, `n_iter=200`, `random_state=42`).
4. **Analyze State Statistics**: Inspect hidden state counts and mean feature values on the original scale.
5. **Visualize Regimes Over Time**: Create an interactive Plotly timeline of detected hidden states.
6. **Analyze Transition Matrix & Duration**: Compute state transition probabilities and expected regime persistence/durations.
7. **Conceptual Comparison & Limitations**: Compare HMM vs. K-Means and document key modeling limitations.

---
### Step 1: Import Required Libraries

**Why we do this:**
- `pandas` & `numpy`: Data structures and numerical computation.
- `plotly.express`: Interactive time series visualization.
- `yfinance`: Downloading market asset data.
- `sklearn.preprocessing.StandardScaler`: Feature standardization.
- `hmmlearn.hmm.GaussianHMM`: Hidden Markov Model with Gaussian emissions.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import yfinance as yf

from sklearn.preprocessing import StandardScaler
from hmmlearn.hmm import GaussianHMM

# Display options
pd.set_option('display.max_columns', None)
print("Libraries successfully imported!")

Libraries successfully imported!


---
### Step 2: Download Market Data & Recreate Feature Matrix

We fetch 5 years of daily closing prices for:
- `^GSPC` (S&P 500 Index)
- `TLT` (iShares 20+ Year Treasury Bond ETF)
- `GLD` (SPDR Gold Shares ETF)

We calculate the six financial features:
1. `Daily_Return`: `Close.pct_change()`
2. `Rolling_Volatility_20`: 20-day rolling std dev of daily returns
3. `Momentum_20`: 20-day percentage price change
4. `Drawdown`: Fractional decline from historical running peak
5. `SP500_TLT_Corr_20`: 20-day rolling correlation between S&P 500 and TLT returns
6. `SP500_GLD_Corr_20`: 20-day rolling correlation between S&P 500 and GLD returns

In [2]:
# Download closing prices
sp500_close = yf.download("^GSPC", period="5y", interval="1d")["Close"]
tlt_close = yf.download("TLT", period="5y", interval="1d")["Close"]
gld_close = yf.download("GLD", period="5y", interval="1d")["Close"]

# Ensure Series format
if isinstance(sp500_close, pd.DataFrame): sp500_close = sp500_close.squeeze()
if isinstance(tlt_close, pd.DataFrame): tlt_close = tlt_close.squeeze()
if isinstance(gld_close, pd.DataFrame): gld_close = gld_close.squeeze()

# Build S&P 500 features
daily_return = sp500_close.pct_change()
vol_20 = daily_return.rolling(window=20).std()
mom_20 = sp500_close.pct_change(periods=20)
peak = sp500_close.cummax()
drawdown = (sp500_close - peak) / peak

# Build Cross-Asset Returns & Correlations
tlt_return = tlt_close.pct_change()
gld_return = gld_close.pct_change()

corr_sp_tlt = daily_return.rolling(window=20).corr(tlt_return)
corr_sp_gld = daily_return.rolling(window=20).corr(gld_return)

# Combine into single feature DataFrame
feature_names = [
    "Daily_Return",
    "Rolling_Volatility_20",
    "Momentum_20",
    "Drawdown",
    "SP500_TLT_Corr_20",
    "SP500_GLD_Corr_20"
]

raw_features_df = pd.DataFrame({
    "Daily_Return": daily_return,
    "Rolling_Volatility_20": vol_20,
    "Momentum_20": mom_20,
    "Drawdown": drawdown,
    "SP500_TLT_Corr_20": corr_sp_tlt,
    "SP500_GLD_Corr_20": corr_sp_gld
})

# Remove NaN rows caused by 20-day rolling windows, preserving Date index
clean_df = raw_features_df[feature_names].dropna().copy()
print(f"Cleaned Feature Matrix Shape: {clean_df.shape}")

[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed

[*********************100%***********************]  1 of 1 completed

Cleaned Feature Matrix Shape: (1235, 6)


---
### Step 3: Standardize Features (`StandardScaler`)

**Why we do this:**
HMM fits a multivariate Gaussian distribution over the feature space for each hidden state. Standardizing features ($z = (x - \mu) / \sigma$) ensures stable optimization and prevents larger scale features from disproportionately influencing state means and covariances.

In [3]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(clean_df[feature_names])
print("Features successfully standardized. Sample scaled shape:", X_scaled.shape)

Features successfully standardized. Sample scaled shape: (1235, 6)


---
### Step 4: Fit Gaussian Hidden Markov Model (HMM)

#### Model Parameters:
- `n_components=4`: 4 unobserved (hidden) market regime states.
- `covariance_type="full"`: Each hidden state learns its own full 6x6 covariance matrix, capturing feature correlations within each state.
- `n_iter=200`: Maximum iterations for the Expectation-Maximization (Baum-Welch) algorithm.
- `random_state=42`: Ensures deterministic initialization and reproducible results.

In [4]:
# Initialize GaussianHMM
hmm_model = GaussianHMM(
    n_components=4,
    covariance_type="full",
    n_iter=200,
    random_state=42
)

# Fit HMM to scaled features
hmm_model.fit(X_scaled)

# Predict hidden state sequence
predicted_states = hmm_model.predict(X_scaled)
clean_df["HMM_State"] = predicted_states

print("=== HMM Fitting Complete ===\n")
print("--- First 10 Rows with HMM States ---")
display(clean_df.head(10))

print("\n--- State Counts (Number of Days per State) ---")
display(clean_df["HMM_State"].value_counts().sort_index())

print("\n--- Feature Means per HMM State (Original Scale) ---")
display(clean_df.groupby("HMM_State")[feature_names].mean())

=== HMM Fitting Complete ===

--- First 10 Rows with HMM States ---


,Daily_Return,Rolling_Volatility_20,Momentum_20,Drawdown,SP500_TLT_Corr_20,SP500_GLD_Corr_20,HMM_State
Date,,,,,,,
2021-10-11,-0.006866,0.009750,-0.024065,-0.026672,-0.135911,-0.322120,0
2021-10-12,-0.002417,0.009696,-0.020797,-0.029025,-0.105195,-0.308855,0
2021-10-13,0.003023,0.009490,-0.026090,-0.026090,-0.067244,-0.207522,0
2021-10-14,0.017063,0.010337,-0.007933,-0.009472,-0.014522,-0.222060,0
2021-10-15,0.007460,0.010262,0.008658,-0.002082,-0.049196,-0.288505,0
2021-10-18,0.003375,0.009413,0.029541,0.000000,0.101640,-0.240841,0
2021-10-19,0.007393,0.009486,0.037996,0.000000,0.060423,-0.218240,0
2021-10-20,0.003664,0.009328,0.031975,0.000000,0.018575,-0.185849,0
2021-10-21,0.002996,0.009003,0.022657,0.000000,0.173971,-0.108886,0



--- State Counts (Number of Days per State) ---


HMM_State
0    576
1    243
2    279
3    137
Name: count, dtype: int64


--- Feature Means per HMM State (Original Scale) ---


,Daily_Return,Rolling_Volatility_20,Momentum_20,Drawdown,SP500_TLT_Corr_20,SP500_GLD_Corr_20
HMM_State,,,,,,
0,0.000794,0.007317,0.019773,-0.009294,0.088455,0.168573
1,0.001715,0.009025,0.040687,-0.067626,0.144492,0.025624
2,-0.001846,0.012299,-0.047160,-0.121037,0.115844,0.195113
3,0.001961,0.016710,0.030969,-0.156628,0.081692,0.335257


---
### Step 5: Visualize HMM States Over Time

We plot the decoded `HMM_State` over time to inspect state persistence and transition dynamics.

In [5]:
plot_hmm_df = clean_df.reset_index()

fig_hmm_ts = px.line(
    plot_hmm_df,
    x="Date",
    y="HMM_State",
    title="S&P 500 Market Regime Sequence (Gaussian HMM, 4 States)",
    labels={"Date": "Date", "HMM_State": "Hidden State (0, 1, 2, 3)"},
    template="plotly_white"
)

fig_hmm_ts.update_traces(mode="lines+markers", marker=dict(size=4))
fig_hmm_ts.update_layout(title_x=0.5, yaxis=dict(dtick=1))
fig_hmm_ts.show()

---
### Step 6: Transition Matrix & State Duration Analysis

#### Understanding the Transition Matrix ($A$):
- The transition matrix entry $A_{ij} = P(S_{t+1} = j \mid S_t = i)$ represents the probability of transitioning from **State $i$** at time $t$ to **State $j$** at time $t+1$.
- Each row $i$ sums to $1.0$.
- **Diagonal elements ($A_{ii}$)**: Represent **state persistence** (the likelihood that the market stays in the same regime state on the next trading day). High diagonal values (e.g., $>0.90$) indicate persistent, sticky market regimes.

#### Expected State Duration Formula:
- Under a first-order Markov process, the expected number of consecutive days spent in State $i$ is given by:
  $$E[D_i] = \frac{1}{1 - A_{ii}}$$
- We also compute the **empirical average run length** observed directly in the decoded state sequence for comparison.

In [6]:
# Extract transition matrix
trans_matrix = hmm_model.transmat_
trans_df = pd.DataFrame(
    trans_matrix,
    index=[f"State {i}" for i in range(4)],
    columns=[f"State {j}" for j in range(4)]
)

print("--- HMM State Transition Matrix (Probability Matrix) ---")
display(trans_df.round(4))

# Calculate theoretical expected duration: 1 / (1 - A_ii)
theoretical_durations = []
for i in range(4):
    p_ii = trans_matrix[i, i]
    exp_dur = 1.0 / (1.0 - p_ii) if (1.0 - p_ii) > 0 else np.nan
    theoretical_durations.append(exp_dur)

# Calculate empirical average duration from decoded sequence
states_seq = clean_df["HMM_State"].values
state_runs = {0: [], 1: [], 2: [], 3: []}

current_state = states_seq[0]
current_len = 1
for s in states_seq[1:]:
    if s == current_state:
        current_len += 1
    else:
        state_runs[current_state].append(current_len)
        current_state = s
        current_len = 1
state_runs[current_state].append(current_len)

empirical_durations = [np.mean(state_runs[i]) if len(state_runs[i]) > 0 else 0 for i in range(4)]

# Combine into summary DataFrame
duration_df = pd.DataFrame({
    "HMM_State": [0, 1, 2, 3],
    "Self_Transition_P_ii": [trans_matrix[i, i] for i in range(4)],
    "Theoretical_Expected_Duration_Days": theoretical_durations,
    "Empirical_Avg_Run_Duration_Days": empirical_durations
})

print("\n--- State Persistence & Average Duration (Trading Days) ---")
display(duration_df.round(2))

--- HMM State Transition Matrix (Probability Matrix) ---


,State 0,State 1,State 2,State 3
State 0,0.9895,0.0000,0.0105,0.0000
State 1,0.0163,0.9713,0.0124,0.0000
State 2,0.0073,0.0218,0.9529,0.0181
State 3,0.0000,0.0074,0.0296,0.9631



--- State Persistence & Average Duration (Trading Days) ---


,HMM_State,Self_Transition_P_ii,Theoretical_Expected_Duration_Days,Empirical_Avg_Run_Duration_Days
0,0,0.99,95.46,82.29
1,1,0.97,34.83,34.71
2,2,0.95,21.22,21.46
3,3,0.96,27.08,27.40


---
### Step 7: Conceptual Comparison: HMM vs. K-Means

| Aspect | K-Means Clustering (Notebook 01) | Hidden Markov Model (Notebook 02) |
| :--- | :--- | :--- |
| **Modeling Concept** | Static geometric partitioning based on instantaneous Euclidean distance in feature space. | Sequential probabilistic modeling of hidden states and feature emission distributions. |
| **Temporal Structure** | Ignores time order; treats each trading day as an independent observation. | Incorporates temporal persistence via a state transition probability matrix ($A_{ij}$). |
| **State Assignments** | Hard assignment to nearest cluster centroid. | Probabilistic state sequence decoding (Viterbi algorithm) taking previous states into account. |
| **Financial Context** | Useful for grouping similar market environments regardless of when they occurred. | Captures market regime stickiness and regime transition probabilities between trading days. |
| **Superiority Claim** | *Neither model is inherently superior without out-of-sample predictive evaluation; they offer complementary analytical views.* |

---
### Step 8: Model Limitations & Practical Considerations

1. **Arbitrary Hidden State Labels**:
   - State integers (`0`, `1`, `2`, `3`) are unlabelled outputs from the Baum-Welch optimization algorithm. They do not automatically carry semantic labels like "Bull" or "Bear". State characteristics must be derived by inspecting feature mean statistics.
2. **Simplifying Statistical Assumptions**:
   - The model assumes a **first-order Markov process** (the next state depends only on the current state) and **multivariate Gaussian emission distributions**.
   - Financial returns often exhibit fat tails (non-Gaussian extreme events) and long memory effects, which simplified HMMs may not fully capture.
3. **In-Sample Fitting & Exploratory Nature**:
   - Fitting the HMM over the entire 5-year dataset is an exploratory, descriptive exercise.
   - To make valid predictive claims or use regimes in trading strategies, strict out-of-sample forward testing (walk-forward optimization) is required to prevent lookahead bias.